In [ ]:
# Importing necessary libraries

import cv2
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
# loding leena image and shwoed using plot

image_path = Path("sample_images/lenna.jpg")
if not image_path.exists():
    raise FileNotFoundError("Place the input image at sample_images/lenna.jpg")
img = cv2.imread(str(image_path))
if img is None:
    raise ValueError(f"Could not load image: {image_path}")
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

plt.imshow(img_rgb)
plt.title('Original Lenna Image')
plt.axis('off')
plt.show()
 
print("Image shape:", img_rgb.shape)

In [ ]:
# detect face and shwoing it

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
faces = face_cascade.detectMultiScale(gray, 1.3, 5)
 
print("Faces detected:", len(faces))
 
# draw face box
face_img = img_rgb.copy()
for (x, y, w, h) in faces:
    cv2.rectangle(face_img, (x, y), (x+w, y+h), (0, 255, 0), 2)
 
plt.imshow(face_img)
plt.title('Face Detection')
plt.axis('off')
plt.show()
 

In [ ]:
# detect eyes, nose and smile

eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')
smile_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_smile.xml')
 
output = img_rgb.copy()
eyes = []
smile = []
 
for (x, y, w, h) in faces:
    cv2.rectangle(output, (x, y), (x+w, y+h), (0, 255, 0), 2)
    cv2.putText(output, 'Face', (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
 
    face_roi_gray = gray[y:y+h, x:x+w]
    face_roi_color = output[y:y+h, x:x+w]
 
    # eyes
    eyes = eye_cascade.detectMultiScale(face_roi_gray, 1.1, 10)
    for (ex, ey, ew, eh) in eyes:
        cv2.rectangle(face_roi_color, (ex, ey), (ex+ew, ey+eh), (255, 0, 0), 2)
        cv2.putText(output, 'Eye', (x+ex, y+ey-5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1)
 
    # nose (estimated from face center)
    nx, ny = x + w//2 - 15, y + h//2 - 10
    cv2.rectangle(output, (nx, ny), (nx+30, ny+25), (255, 255, 0), 2)
    cv2.putText(output, 'Nose', (nx, ny-5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)
 
    # smile
    lower = face_roi_gray[h//2:, :]
    lower_color = face_roi_color[h//2:, :]
    smile = smile_cascade.detectMultiScale(lower, 1.8, 20)
    for (sx, sy, sw, sh) in smile:
        cv2.rectangle(lower_color, (sx, sy), (sx+sw, sy+sh), (0, 255, 255), 2)
        cv2.putText(output, 'Smile', (x+sx, y+h//2+sy-5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)
 
print("Eyes detected:", len(eyes))
print("Nose: estimated from face center")
print("Smile detected:", len(smile))
 
plt.imshow(output)
plt.title('Eyes, Nose and Smile Detection')
plt.axis('off')
plt.show()
 

In [ ]:
# ploting facial landmarks

landmark_img = img_rgb.copy()
landmarks = {}
 
if len(faces) > 0:
    x, y, w, h = faces[0]
    landmarks = {
        'Left Eye':    (x + int(w*0.30), y + int(h*0.35)),
        'Right Eye':   (x + int(w*0.70), y + int(h*0.35)),
        'Nose Tip':    (x + int(w*0.50), y + int(h*0.55)),
        'Left Mouth':  (x + int(w*0.35), y + int(h*0.75)),
        'Right Mouth': (x + int(w*0.65), y + int(h*0.75)),
        'Chin':        (x + int(w*0.50), y + int(h*0.92)),
        'Left Cheek':  (x + int(w*0.15), y + int(h*0.55)),
        'Right Cheek': (x + int(w*0.85), y + int(h*0.55)),
        'Forehead':    (x + int(w*0.50), y + int(h*0.12)),
    }
    for name, (lx, ly) in landmarks.items():
        cv2.circle(landmark_img, (lx, ly), 5, (255, 0, 0), -1)
        cv2.putText(landmark_img, name.split()[0], (lx+5, ly), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (255, 255, 255), 1)
 
print("Landmarks plotted:", len(landmarks))
 
plt.imshow(landmark_img)
plt.title('Facial Landmarks (9 points)')
plt.axis('off')
plt.show()
 

### Important limitation

The CNN section is a learning demonstration, not a trained facial-classification model. It creates augmented copies of one image and pairs them with randomly generated targets, so its learned feature vector and loss do not represent validated facial predictions. The eye detection uses a Haar cascade; the nose position and facial landmarks are estimated geometrically.


In [ ]:
# now build the CNN model

model = models.Sequential([
    layers.Input(shape=(64, 64, 3)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu')
])
 
model.compile(optimizer='adam', loss='mse')
model.summary()
 

In [ ]:
# training the CNN

img_resized = cv2.resize(img_rgb, (64, 64))
 
X = []
for i in range(100):
    aug = img_resized.copy().astype(np.float32)
    aug = aug * np.random.uniform(0.8, 1.2)
    if np.random.rand() > 0.5:
        aug = aug[:, ::-1, :]
    aug = np.clip(aug / 255.0, 0, 1)
    X.append(aug)
 
X = np.array(X)
y = np.random.rand(100, 64)
 
print("Training CNN model...")
history = model.fit(X, y, epochs=5, batch_size=16, verbose=1)
print("Training complete!")
 
plt.plot(history.history['loss'], marker='o', color='blue')
plt.title('CNN Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

In [ ]:
# Extract features from Lenna using the trained CNN

input_img = img_resized.astype(np.float32) / 255.0
input_img = np.expand_dims(input_img, axis=0)
 
features = model.predict(input_img)
print("Feature vector shape:", features.shape)
 
plt.bar(range(len(features[0])), features[0], color='steelblue')
plt.title('CNN Feature Vector (64 features)')
plt.xlabel('Feature Index')
plt.ylabel('Value')
plt.show()

In [ ]:
# Illustrative image heuristics (not a validated gender or age model)

hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
saturation = np.mean(hsv[:, :, 1])
texture = cv2.Laplacian(gray, cv2.CV_64F).var()
 
score = 0
if saturation > 60:
    score += 1
if texture < 500:
    score += 1
 
gender = "Female" if score >= 1 else "Male"
 
if texture > 400:
    age_group = "Teen (13-19)"
elif texture > 200:
    age_group = "Young Adult (20-35)"
else:
    age_group = "Adult (36-50)"
 
print("----- Facial Analysis -----")
print("Gender    :", gender)
print("Age Group :", age_group)
print("Saturation:", round(saturation, 2))
print("Texture   :", round(texture, 2))

In [ ]:
# final results - all in one 

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
 
axes[0].imshow(output)
axes[0].set_title('Detected Features')
axes[0].axis('off')
 
axes[1].imshow(landmark_img)
axes[1].set_title('Facial Landmarks')
axes[1].axis('off')
 
axes[2].axis('off')
result = (
    f"  ----- Results -----\n\n"
    f"  Gender     : {gender}\n"
    f"  Age Group  : {age_group}\n\n"
    f"  Faces      : {len(faces)}\n"
    f"  Eyes       : {len(eyes)}\n"
    f"  Nose       : 1\n"
    f"  Smile      : {len(smile)}\n"
    f"  Landmarks  : {len(landmarks)}"
)
axes[2].text(0.05, 0.5, result, fontsize=12, verticalalignment='center',
             fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='lightyellow'))
axes[2].set_title('Analysis Summary')
 
plt.tight_layout()
plt.show()